***Construct Next-Token Prediction Targets***

In [ ]:
def make_input_target(token_ids, context_size):
	"""Build next-token prediction (input, target) pair.

	Args:
		token_ids: list[int] of token IDs (length >= context_size + 1)
		context_size: int, number of tokens in the input window
	Returns:
		list: [x, y] where x and y are lists of length context_size,
			  with y being x shifted right by one position.
	"""
	if len(token_ids) < context_size + 1:
		raise ValueError(
			f"token_ids too short: got {len(token_ids)} tokens, "
			f"need at least {context_size + 1}"
		)

	if context_size == 0 :
		return [[],[]]

	x = token_ids[:context_size]
	y = token_ids[1: context_size + 1]
	return [x, y]

print(make_input_target([7, 8], 0))

print(make_input_target([40, 367, 2885, 1464, 1807, 3619], 4))

[[], []]
[[40, 367, 2885, 1464], [367, 2885, 1464, 1807]]


***Token Embedding Lookup Table***


In [ ]:
import numpy as np

def token_embedding_lookup(vocab_size: int, embed_dim: int, token_ids: list, seed: int = 0) -> list:
	"""
	Build a random embedding table of shape (vocab_size, embed_dim) using
	np.random.default_rng(seed).standard_normal(...), then return the rows
	corresponding to token_ids as a nested list.
	"""
	rng = np.random.default_rng(seed)
	E = rng.standard_normal((vocab_size, embed_dim))
	# print(E.shape)     # should print (5, 3)
	# print(E[0])        # should start with 0.1257..., -0.1321..., 0.6404...
	return E[token_ids].tolist()


out = token_embedding_lookup(5, 3, [0, 2, 4], seed=0)
print(np.round(out, 4).tolist())


[[0.1257, -0.1321, 0.6404], [1.304, 0.9471, -0.7037], [-2.325, -0.2188, -1.2459]]


***Embedding Layer as One-Hot Matrix Multiplication***

In [ ]:
import numpy as np

def embedding_via_one_hot(token_ids, W):
	"""
	Compute token embeddings via one-hot encoding and matrix multiplication.

	Args:
		token_ids: list or 1D array of integer token IDs
		W: numpy array of shape (vocab_size, embed_dim)

	Returns:
		numpy array of shape (len(token_ids), embed_dim)
	"""
	num_tokens = len(token_ids)     # 3
	vocab_size = W.shape[0]  # 4
	H = np.zeros((num_tokens, vocab_size))

	for i in range(num_tokens):
		H[i, token_ids[i]] = 1

	return H @ W

W = np.array([[1.0, 2.0, 3.0],
			  [4.0, 5.0, 6.0],
			  [7.0, 8.0, 9.0],
			  [10.0, 11.0, 12.0]])
print(embedding_via_one_hot([2, 0, 3], W).tolist())

[[7.0, 8.0, 9.0], [1.0, 2.0, 3.0], [10.0, 11.0, 12.0]]


**Reshape Matrix**


In [ ]:
import numpy as np

def reshape_matrix(a: list[list[int|float]], new_shape: tuple[int, int]) -> list[list[int|float]]:
	#Write your code here and return a python list after reshaping by using numpy's tolist() method
	a = np.array(a)
	elements  = len(a) * len(a[0])
	new_shapee = new_shape[0] * new_shape[1]
	if elements != new_shapee :
		return []
	else :
		return a.reshape(new_shape[0],new_shape[1]).tolist()

print(reshape_matrix([[1, 2, 3, 4], [5, 6, 7, 8]], (1, 4)))
print(reshape_matrix([[1,2,3],[4,5,6]], (3, 2)))


[]
[[1, 2], [3, 4], [5, 6]]


In [ ]:
import numpy as np

def reshape_matrix(a: list[list[int|float]], new_shape: tuple[int, int]) -> list[list[int|float]]:
	#Write your code here and return a python list after reshaping by using numpy's tolist() method
	a = np.array(a)
	p_old , q_old = len(a) , len(a[0])
	p_new , q_new = new_shape[0] , new_shape[1]
	if p_old * q_old != p_new * q_new:
		return []
	else:
		return a.reshape(p_new,q_new).tolist()



***Add a Bias Vector to a Batch via Broadcasting***

In [ ]:
import torch

def add_bias(x: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
	# TODO: add b to every row of x using broadcasting
	return x + b 

import torch
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
b = torch.tensor([10.0, 20.0, 30.0])
print(add_bias(x, b).tolist())

[[11.0, 22.0, 33.0], [14.0, 25.0, 36.0]]


***Softmax Activation Function Implementation***

In [ ]:
import math
import numpy as np 
def softmax(scores):
	maxx = np.max(scores)     
	denom = np.sum([math.exp(z - maxx) for z in scores])       # compute once, using ALL scores
	
	results = []
	for score in scores:
		result = float(math.exp(score - maxx) / denom )   # now use maxx and denom here
		results.append(result)
	return results

print([round(x, 4) for x in softmax([1, 2, 3])])

[0.09, 0.2447, 0.6652]


In [ ]:
# using Numpy

import math
import numpy as np 
def softmax(scores):
	scores = np.array(scores)
	maxx = np.max(scores)     
	exp_scores = np.exp(scores - maxx)  
	denom = np.sum(exp_scores)
	return (exp_scores / denom)

print([float(round(x, 4)) for x in softmax([1, 2, 3])])

[0.09, 0.2447, 0.6652]


***MLP Character-Level Language Model Forward Pass***

In [ ]:
import numpy as np

def mlp_forward_loss(X, Y, C, W1, b1, W2, b2):
	
	# X.shape = (N, block_size)
	# Y.shape = (N,)
	# C.shape = (Vocab_size, emb_dim)

	emb = C[X]        # emb.shape = (N, block_size, emb_dim)
	N, block_size, emb_dim = emb.shape

	# flatten
	flat  = emb.reshape(-1, block_size * emb_dim)    # flat.shape = (N, block_size * emb_dim)

	# W1.shape = (block_size * emb_dim, hidden) 
	h = np.tanh(flat @ W1 + b1)   # flat.shape = (32, 30) and W1.shape = (30, 32) --> (flat @ W1).shape = (32, 32) + b1.shape = (32,) = (32, 32)

	# Computes output logits:h.shape = (N, hidden) || W2.shape = (hidden, vocab_size) || b2.shape = (vocab_size,) || logits.shape = (N, vocab_size)
	logits = h @ W2 + b2  # scores 
	maxx = np.max(logits, axis=1, keepdims=True)
	exp_scores = np.exp(logits - maxx)    # exp_scores.shape = (32,27)
	denom = np.sum(exp_scores, axis=1, keepdims=True)
	probs = exp_scores / denom
	# print(probs.sum(axis=1))   # each row should be = 1
	
	# Cross Entropy Loss 
	correct_probs = probs[range(N), Y]  # advanced indexing
	# likelihood 
	loss = -np.log(correct_probs).mean()
	return loss

import numpy as np
X = np.array([[0,1],[1,2]])
Y = np.array([2,0])
C = np.array([[0.1,0.2],[0.3,0.4],[0.5,0.6]])
W1 = np.ones((4,3))*0.1
b1 = np.zeros(3)
W2 = np.ones((3,3))*0.1
b2 = np.zeros(3)
print(round(mlp_forward_loss(X, Y, C, W1, b1, W2, b2), 6))

1.098612


***Numerically Stable Cross-Entropy***

In [ ]:
import torch

def cross_entropy(logits, targets):
	log_z = torch.logsumexp(logits, dim=1)

	correct_logits = logits[
		torch.arange(logits.shape[0]),
		targets
	]

	loss = -correct_logits + log_z

	return loss.mean()


logits = torch.tensor([[2.0, 1.0, 0.0]])
targets = torch.tensor([0])
print(round(cross_entropy(logits, targets).item(), 4))

0.4076


***Batch Iterator for Dataset***

In [ ]:
import numpy as np

def batch_iterator(X, y=None, batch_size=64):
	# Your code here
	n = len(X)   # n = 5
	result = []  # [[[[1, 2], [3, 4]],[1,2]], [[[5, 6], [7, 8]],[3, 4]]] ...
	for i in range(0, n, batch_size):   # range(0,5,2) || i = 0 || i = 2 || i = 4 
		X_batch = X[i:i+batch_size]     # X_batch = X[0:2] --> [[1, 2], [3, 4]] || X_batch = X[2:4] --> [[5, 6], [7, 8]]
		
		if y is None:
			result.append(list(X_batch))
		else :
			y_batch = y[i:i+batch_size]    # y_batch = y[0:2] --> [1,2] || y_batch = y[2:4] --> [3, 4]
			result.append([X_batch.tolist(), y_batch.tolist()]) 
	return result 		
					

print(batch_iterator(np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]]), np.array([1, 2, 3, 4, 5]), batch_size=2))

[[[[1, 2], [3, 4]], [1, 2]], [[[5, 6], [7, 8]], [3, 4]], [[[9, 10]], [5]]]


***PyTorch: Implement Your Own Gradient Descent Training Step***



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def train_step(model, x_batch, y_batch, lr):

	# 1. Zero gradients
	for param in model.parameters():
		if param.grad is not None:
			param.grad.zero_()

	# 2. Forward
	logits = model(x_batch)

	# 3. Loss
	loss = F.cross_entropy(logits, y_batch)

	# 4. Backpropagation
	loss.backward()

	# 5. Update weights
	with torch.no_grad():
		for param in model.parameters():
			param -= lr * param.grad

	# 6. Return Python float
	return loss.item()



***Learning Rate Range Finder for Linear Regression***

In [4]:
import numpy as np

def lr_range_finder(X, y, w0, a, b, n_steps):
	"""
	Sweep learning rate from 10**a to 10**b over n_steps full-batch GD updates
	on a linear regression model (no bias). Return the list of MSE losses after
	each update.
	"""
	losses = []
	w = w0
	for k in range(n_steps):
		if n_steps > 1 :
			pw = a + (((b-a)*k)/(n_steps - 1))
			lr = 10**pw
		else :
			lr = 10**a
		# Prediction
		pred = X @ w
		# Error
		error = pred - y
		n = len(X)
		# Gradient
		gradient = (2/n) * X.T @ error 
		# Update
		w = w - lr * gradient
		# new prediction 
		new_pred = X @ w
		# loss 
		loss = np.mean((new_pred - y) ** 2)
		losses.append(float(loss))

	return losses

						

X = np.array([[1.0],[2.0],[3.0],[4.0]])
y = np.array([2.0, 4.0, 6.0, 8.0])
w0 = np.array([0.0])
print([round(v, 6) for v in lr_range_finder(X, y, w0, a=-2, b=-1, n_steps=3)])
print("-----------------------------------------")
X = np.array([[1.0, 2.0],[2.0, 1.0],[3.0, 0.5],[4.0, 2.5]])
y = np.array([5.0, 4.0, 4.0, 9.0])
w0 = np.zeros(2)
print([round(v, 6) for v in lr_range_finder(X, y, w0, a=-3, b=0, n_steps=4)])

[21.675, 5.989165, 1.497291]
-----------------------------------------
[33.21178, 21.913336, 19.225628, 6251.79245]
